# L-1 Step 3：Self-Consistency LLM 分類

對 Step 2 未定案（MEDIUM，similarity 0.50–0.65）送 LLM 三次取多數決。
- 3/3 同意 → **CONFIRM_HIGH**
- 2/3 同意 → **CONFIRM_MED**
- 1/3 或全不同 → **UNCERTAIN**（降權，不進入最終分類）

**執行前提**：Step 2 已完成（`step2_results.jsonl` 存在）。


## 0. 安裝套件

In [1]:
!pip install google-genai pyodbc pandas tqdm

## 1. 匯入套件 & 全域設定

In [2]:
import os, re, json, time, configparser
from pathlib import Path
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd

# ── 路徑設定 ──────────────────────────────────────────
BASE_DIR    = Path(r"D:\yujui\痛點需求地圖")
STEP2_DIR   = BASE_DIR / "step2_output"
OUTPUT_DIR  = BASE_DIR / "step3_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STEP2_JSONL   = STEP2_DIR / "step2_results.jsonl"
MATCHED_JSONL = STEP2_DIR / "_a2_matched.jsonl"   # Step2 已補回 log_content
RESULT_JSONL  = OUTPUT_DIR / "step3_results.jsonl"

# ── LLM 設定 ──────────────────────────────────────────
MODEL          = "gemini-2.0-flash"
TEMP           = 0.4      # 適度隨機，確保 3 次有差異性
N_VOTES        = 3        # Self-Consistency 投票次數
RPM_LIMIT      = 14       # 保守低於免費層 15 RPM
SLEEP_PER_CALL = 60 / RPM_LIMIT   # ≈ 4.3 秒

LAYERS = ["L1","L2","L3","L4","L5","L6","L7"]

print("全域設定載入完成")
print(f"  STEP2_JSONL  : {STEP2_JSONL}")
print(f"  OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"  MODEL={MODEL}  TEMP={TEMP}  N_VOTES={N_VOTES}")
print(f"  RPM={RPM_LIMIT}  → {SLEEP_PER_CALL:.1f} 秒/次呼叫")


d:\miniforge3\envs\yujui_even\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


全域設定載入完成
  STEP2_JSONL  : D:\yujui\痛點需求地圖\step2_output\step2_results.jsonl
  OUTPUT_DIR   : D:\yujui\痛點需求地圖\step3_output
  MODEL=gemini-2.0-flash  TEMP=0.4  N_VOTES=3
  RPM=14  → 4.3 秒/次呼叫


## 2. 載入設定 & Gemini 連線

In [3]:
from google import genai

GCP_INI = r"D:\yujui\GoogleCloud.ini"
cfg = configparser.ConfigParser()
cfg.read(GCP_INI, encoding="utf-8")

# GoogleCloud.ini key 名稱帶單引號，掃描所有 section
def _get_ini(cfg, key):
    for sec in list(cfg.sections()) + ['DEFAULT']:
        for k in (key, f"'{key}'"):
            try:
                v = cfg.get(sec, k)
                if v: return v.strip("'\" ")
            except: pass
    raise KeyError(f'{key} not found in {GCP_INI}')

PROJECT_ID = _get_ini(cfg, 'project_id')
GEMINI_KEY = _get_ini(cfg, 'gemini_api_key')

client = genai.Client(api_key=GEMINI_KEY)
print(f"Gemini 連線成功  project={PROJECT_ID}")

Gemini 連線成功  project=digiwin-ai


---
## 子任務 A：讀取 Step 2 MEDIUM 待分類資料

從 `step2_results.jsonl` 取出 `step2_status == MEDIUM` 的記錄，
再從 `_a2_matched.jsonl`（已有 log_content）join 補回原始文字。
**不需重查 SQL**。


In [4]:
# ── A1：統計 Step 2 各 status 筆數 ─────────────────────
stats = Counter()
with open(STEP2_JSONL, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        stats[json.loads(line)["step2_status"]] += 1

total = sum(stats.values())
print(f"Step 2 結果統計（共 {total:,} 筆）：")
for k in ["HIGH","MEDIUM","LOW"]:
    n = stats.get(k, 0)
    print(f"  {k:8s} {n:>10,} 筆  ({n/total*100:.1f}%)")
print()
print(f"Step 3 待處理：{stats.get('MEDIUM',0):,} 筆（MEDIUM）")
n_medium = stats.get("MEDIUM", 0)
est_h = n_medium * SLEEP_PER_CALL * N_VOTES / 3600
print(f"預估耗時（免費層 {RPM_LIMIT} RPM）：{est_h:.1f} 小時")
print(f"建議：使用付費 API 提高 RPM 可大幅縮短（調高 RPM_LIMIT 並降低 SLEEP_PER_CALL）")


Step 2 結果統計（共 935,570 筆）：
  HIGH        935,556 筆  (100.0%)
  MEDIUM           14 筆  (0.0%)
  LOW               0 筆  (0.0%)

Step 3 待處理：14 筆（MEDIUM）
預估耗時（免費層 14 RPM）：0.1 小時
建議：使用付費 API 提高 RPM 可大幅縮短（調高 RPM_LIMIT 並降低 SLEEP_PER_CALL）


In [5]:
# ── A2：載入 MEDIUM 記錄 + 從 _a2_matched.jsonl 補回 log_content ──
import gc

# 讀 Step2 MEDIUM event_ids
medium_records = {}   # event_id → meta dict
with open(STEP2_JSONL, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        if r["step2_status"] == "MEDIUM":
            medium_records[r["event_id"]] = {
                "company_id":       r.get("company_id",""),
                "ym":               r.get("ym",""),
                "step2_confidence": r.get("step2_confidence", 0.0),
                "step2_result":     r.get("step2_result",""),
            }

print(f"Step 2 MEDIUM：{len(medium_records):,} 筆")

# 從 _a2_matched.jsonl 補回 log_content（in-memory join，無需重查 SQL）
matched = {}
need_ids = set(medium_records.keys())
with open(MATCHED_JSONL, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        eid = r["event_id"]
        if eid in need_ids:
            matched[eid] = {"log_content": r["log_content"], "log_len": r.get("log_len",0)}
            if len(matched) == len(need_ids):
                break   # 全找到提早退出

print(f"log_content 補回：{len(matched):,} 筆  |  未找到：{len(need_ids)-len(matched):,} 筆")

# 合併成 todo_df
rows = []
for eid, meta in medium_records.items():
    lc = matched.get(eid, {})
    rows.append({
        "event_id":         eid,
        "company_id":       meta["company_id"],
        "ym":               meta["ym"],
        "step2_confidence": meta["step2_confidence"],
        "step2_result":     meta["step2_result"],
        "log_content":      lc.get("log_content",""),
        "log_len":          lc.get("log_len", 0),
    })

todo_df = pd.DataFrame(rows)
del medium_records, matched, rows
gc.collect()

print(f"todo_df 建立完成：{len(todo_df):,} 筆")
todo_df.head(2)


Step 2 MEDIUM：14 筆
log_content 補回：14 筆  |  未找到：0 筆
todo_df 建立完成：14 筆


,event_id,company_id,ym,step2_confidence,step2_result,log_content,log_len
0,262c06c2b0fd8aee,0000129354,2014-01,0.6147,L3,代理OZOC、INDIVI、UNTITLED、THE EMPORIUM、Reflect、AD...,67
1,7df2ffc8754a169e,0000130548,2013-04,0.6435,L4,朱建榮【流通服務業-共同精進管理系列 觀光發展趨勢之策略維新與換軌突維之路(台北)】已於20...,52


---
## 子任務 B：LLM Self-Consistency 分類

每筆日報送 LLM `N_VOTES`（3）次，溫度 0.4，取多數決。

| 一致性 | Status | 說明 |
|--------|--------|------|
| 3/3 | CONFIRM_HIGH | 最高信心 |
| 2/3 | CONFIRM_MED | 多數決通過 |
| 1/3 或 0/3 | UNCERTAIN | 降權，不進最終分類 |


In [6]:
# ── B1：Prompt + 呼叫 + 投票函數 ────────────────────────
from google.genai import types as genai_types

SYSTEM_PROMPT = """你是業務日誌 L 層分類專家。根據以下定義，判斷這篇業務日誌最符合哪個 L 層：

L1 起因/阻礙：客戶面臨的問題、卡關、困難（如製造能力不足、流程障礙、風險）
L2 角色壓力：組織內部壓力，老闆/主管要求、KPI、部門績效考核
L3 戰略目標：公司長期轉型方向、三年計畫、策略佈局（如數位轉型、降本增效）
L4 產業議題：外部環境討論，產業趨勢、法規政策、競爭態勢、ESG
L5 問項精煉：客戶正在評估方案、比較選項、確認需求細節（如想了解、能不能）
L6 動態屬性：客戶態度/溫度有變化，如猶豫轉積極、比上次更保守、突然表態
L7 實戰對策：成交/結案階段，合約、簽訂、付款、方案確認、折扣談判

請只回傳 JSON，格式：
{"l_layer": "L1", "confidence": 0.85, "reasoning": "30字內說明"}
若無法判斷，回傳：{"l_layer": "uncertain", "confidence": 0.0, "reasoning": "無法判斷原因"}
"""

_JSON_PAT = re.compile(r'\{[^{}]+\}', re.DOTALL)

def call_llm_once(log_content: str) -> tuple:
    """回傳 (l_layer, confidence)，失敗回傳 ('uncertain', 0.0)"""
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"以下是業務日誌，請判斷 L 層：\n\n{log_content[:500]}",
            config=genai_types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=TEMP,
                response_mime_type="application/json",
            ),
        )
        text = resp.text.strip()
        m = _JSON_PAT.search(text)
        if not m:
            return "uncertain", 0.0
        data = json.loads(m.group())
        layer = data.get("l_layer", "uncertain")
        if layer not in LAYERS:
            layer = "uncertain"
        return layer, round(float(data.get("confidence", 0.0)), 4)
    except Exception as e:
        return "uncertain", 0.0


def sc_classify(log_content: str, n: int = N_VOTES) -> dict:
    """送 LLM n 次，取多數決。回傳 {l_layer, confidence, agreement, votes}"""
    votes = []
    for _ in range(n):
        layer, conf = call_llm_once(log_content)
        votes.append((layer, conf))
        time.sleep(SLEEP_PER_CALL)

    valid = [(l, c) for l, c in votes if l != "uncertain"]
    if not valid:
        return {"l_layer":"uncertain","confidence":0.0,"agreement":"0/3","votes":[v[0] for v in votes]}

    layer_counts = Counter(l for l, _ in valid)
    top_layer, top_count = layer_counts.most_common(1)[0]

    if top_count >= 2:
        avg_conf = sum(c for l, c in valid if l == top_layer) / top_count
        return {"l_layer":top_layer,"confidence":round(avg_conf,4),
                "agreement":f"{top_count}/{n}","votes":[v[0] for v in votes]}
    else:
        return {"l_layer":"uncertain","confidence":0.0,"agreement":"0/3","votes":[v[0] for v in votes]}


def get_status(agreement: str) -> str:
    if agreement == "3/3": return "CONFIRM_HIGH"
    if agreement == "2/3": return "CONFIRM_MED"
    return "UNCERTAIN"


print("B1 函數定義完成")
print(f"  SLEEP_PER_CALL = {SLEEP_PER_CALL:.1f} 秒  →  每筆耗時 ≈ {SLEEP_PER_CALL*N_VOTES:.0f} 秒")


B1 函數定義完成
  SLEEP_PER_CALL = 4.3 秒  →  每筆耗時 ≈ 13 秒


In [7]:
# ── B2：批次 SC 分類（斷點續跑）────────────────────────
# RESUME = True  → 從上次中斷處繼續
# RESUME = False → 從頭重跑（清空舊結果）

RESUME = True

if not RESUME and RESULT_JSONL.exists():
    os.remove(RESULT_JSONL)
    print(f"已清除舊結果：{RESULT_JSONL}")

done_ids = set()
if RESUME and RESULT_JSONL.exists():
    with open(RESULT_JSONL, encoding="utf-8") as f:
        done_ids = {json.loads(l)["event_id"] for l in f if l.strip()}
    print(f"斷點續跑：已完成 {len(done_ids):,} 筆")

run_df = todo_df[~todo_df["event_id"].isin(done_ids)].reset_index(drop=True)
total  = len(run_df)
print(f"待分類：{total:,} 筆")

written = errors = 0

with open(RESULT_JSONL, "a", encoding="utf-8") as fout:
    for _, row in tqdm(run_df.iterrows(), total=total, desc="SC 分類"):
        try:
            result = sc_classify(row["log_content"])
        except Exception as e:
            print(f"[錯誤] event_id={row['event_id']}: {e}")
            errors += 1
            continue

        fout.write(json.dumps({
            "event_id":         row["event_id"],
            "company_id":       row.get("company_id", ""),
            "ym":               row.get("ym", ""),
            "step2_confidence": row.get("step2_confidence", 0.0),
            "step3_result":     result["l_layer"],
            "step3_confidence": result["confidence"],
            "step3_status":     get_status(result["agreement"]),
            "step3_agreement":  result["agreement"],
            "votes":            result["votes"],
            "classified_at":    datetime.now().isoformat(),
        }, ensure_ascii=False) + "\n")
        written += 1

print(f"\n分類完成：{written:,} 筆 → {RESULT_JSONL}")
if errors:
    print(f"⚠️  錯誤：{errors} 筆（已跳過）")


待分類：14 筆


SC 分類: 100%|██████████| 14/14 [03:42<00:00, 15.93s/it]


分類完成：14 筆 → D:\yujui\痛點需求地圖\step3_output\step3_results.jsonl


---
## 子任務 E：信心分布報告


In [8]:
# ── E1：Step 3 結果統計 ──────────────────────────────────
results3 = [json.loads(l) for l in open(RESULT_JSONL, encoding="utf-8") if l.strip()]
df3 = pd.DataFrame(results3)

total3 = len(df3)
print(f"Step 3 分類結果（共 {total3:,} 筆）：")

print("\n── Status 分布 ─────────────────────")
for s in ["CONFIRM_HIGH","CONFIRM_MED","UNCERTAIN"]:
    n = (df3["step3_status"] == s).sum()
    print(f"  {s:<15s} {n:>8,} 筆  ({n/total3*100:.1f}%)")

print("\n── 各 L 層分布（CONFIRM）───────────")
confirmed = df3[df3["step3_status"].isin(["CONFIRM_HIGH","CONFIRM_MED"])]
for layer in ["L1","L2","L3","L4","L5","L6","L7"]:
    n = (confirmed["step3_result"] == layer).sum()
    print(f"  {layer}  {n:>8,} 筆")

print("\n── 一致性分析 ───────────────────────")
for ag in ["3/3","2/3","1/3","0/3"]:
    n = (df3["step3_agreement"] == ag).sum()
    print(f"  agreement={ag}  {n:>8,} 筆  ({n/total3*100:.1f}%)")


Step 3 分類結果（共 14 筆）：

── Status 分布 ─────────────────────
  CONFIRM_HIGH          10 筆  (71.4%)
  CONFIRM_MED            1 筆  (7.1%)
  UNCERTAIN              3 筆  (21.4%)

── 各 L 層分布（CONFIRM）───────────
  L1         7 筆
  L2         0 筆
  L3         0 筆
  L4         3 筆
  L5         1 筆
  L6         0 筆
  L7         0 筆

── 一致性分析 ───────────────────────
  agreement=3/3        10 筆  (71.4%)
  agreement=2/3         1 筆  (7.1%)
  agreement=1/3         0 筆  (0.0%)
  agreement=0/3         3 筆  (21.4%)


In [9]:
# ── E2：全量分類合併（Step1 HIGH + Step2 HIGH + Step3 CONFIRM）──
STEP1_JSONL = BASE_DIR / "step1_output" / "step1_results.jsonl"
FINAL_CSV   = OUTPUT_DIR / "phase_l_final.csv"

rows_final = []

# Step 1 HIGH
with open(STEP1_JSONL, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        if r["step1_confidence"] == "HIGH":
            rows_final.append({"event_id":r["event_id"],"company_id":r.get("company_id",""),
                "ym":r.get("ym",""),"l_layer":r["step1_result"],"confidence":1.0,"source":"step1"})
s1 = len(rows_final)
print(f"Step 1 HIGH：{s1:,} 筆")

# Step 2 HIGH
with open(STEP2_JSONL, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        if r["step2_status"] == "HIGH":
            rows_final.append({"event_id":r["event_id"],"company_id":r.get("company_id",""),
                "ym":r.get("ym",""),"l_layer":r["step2_result"],
                "confidence":r.get("step2_confidence",0.0),"source":"step2"})
s2 = len(rows_final) - s1
print(f"Step 2 HIGH：{s2:,} 筆")

# Step 3 CONFIRM
for r in results3:
    if r["step3_status"] in ("CONFIRM_HIGH","CONFIRM_MED"):
        rows_final.append({"event_id":r["event_id"],"company_id":r.get("company_id",""),
            "ym":r.get("ym",""),"l_layer":r["step3_result"],
            "confidence":r.get("step3_confidence",0.0),"source":"step3"})
s3 = len(rows_final) - s1 - s2
print(f"Step 3 CONFIRM：{s3:,} 筆")

df_final = pd.DataFrame(rows_final)
df_final.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")

total_f = len(df_final)
print(f"\n全量合併：{total_f:,} 筆 → {FINAL_CSV}")
print("\n── 各 L 層最終覆蓋 ─────────────────────")
for layer in ["L1","L2","L3","L4","L5","L6","L7"]:
    n = (df_final["l_layer"] == layer).sum()
    print(f"  {layer}  {n:>8,} 筆")


Step 1 HIGH：867,023 筆
Step 2 HIGH：935,556 筆
Step 3 CONFIRM：11 筆

全量合併：1,802,590 筆 → D:\yujui\痛點需求地圖\step3_output\phase_l_final.csv

── 各 L 層最終覆蓋 ─────────────────────
  L1   461,267 筆
  L2   366,442 筆
  L3   183,624 筆
  L4    56,679 筆
  L5   433,565 筆
  L6   114,567 筆
  L7   186,410 筆
